**Q1**
So here is the understanding . there are 3 layers in the NN. first which takes input 784 pixel. second which has 128 or 64 nodes and each node takes eack input into a n activation func (input as linear ). next layer has 10 nodes each represent the digit. we have weights and biases in each layer ... we initialize them randomly..and we calculate  the values at each nodes...then we calculate error...tghen from this values we find the change in error with respect to each parameter..using chain rule and inputing values then we use this derivative to correct our weight and biases using learning rate.



In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# ==========================================
# 1. ACTIVATION FUNCTIONS & LOSS HELPERS
# ==========================================

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return Z > 0

def softmax(Z):
    # Subtracting np.max prevents numerical overflow (exploding exponentials)
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def compute_loss(Y_pred, Y_true):
    """Categorical Cross-Entropy Loss"""
    m = Y_true.shape[0]
    Y_pred = np.clip(Y_pred, 1e-15, 1 - 1e-15)  # Avoid log(0)
    loss = -np.sum(Y_true * np.log(Y_pred)) / m
    return loss

def one_hot_encode(Y, num_classes=10):
    m = Y.shape[0]
    one_hot = np.zeros((m, num_classes))
    one_hot[np.arange(m), Y] = 1
    return one_hot

def get_accuracy(Y_pred, Y_true_labels):
    predictions = np.argmax(Y_pred, axis=1)
    return np.mean(predictions == Y_true_labels)

# ==========================================
# 2. MULTI-LAYER PERCEPTRON CLASS
# ==========================================

class MLPFromScratch:
    def __init__(self, input_size=784, hidden_size=128, output_size=10, learning_rate=0.1):
        self.lr = learning_rate

        # He (Kaiming) Initialization for the hidden layer
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))

        # Xavier (Glorot) Initialization for the output layer
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        # Layer 1 (Input to Hidden)
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = relu(self.Z1)

        # Layer 2 (Hidden to Output)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = softmax(self.Z2)

        return self.A2

    def backward(self, X, Y_true):
        m = X.shape[0]

        # Output layer error
        dZ2 = self.A2 - Y_true
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        # Hidden layer error
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * relu_derivative(self.Z1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Gradient Descent Step
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

# ==========================================
# 3. DATA LOADING & EXECUTION PIPELINE
# ==========================================

print("Fetching MNIST dataset (this might take a minute)...")
# Fetch MNIST via openml (70,000 images, 28x28 pixels flattened into 784 features)
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, Y = mnist["data"], mnist["target"].astype(int)

# Normalize pixel values from [0, 255] to [0, 1] for stable gradient updates
X = X / 255.0

# Split into Training (60k) and Testing (10k) sets
X_train, X_test, Y_train_labels, Y_test_labels = train_test_split(
    X, Y, test_size=10000, random_state=42
)

# Convert integer targets to one-hot encodings for cross-entropy loss calculation
Y_train_onehot = one_hot_encode(Y_train_labels)
Y_test_onehot = one_hot_encode(Y_test_labels)

# Initialize network configuration
num_samples = X_train.shape[0]
mlp = MLPFromScratch(input_size=784, hidden_size=128, output_size=10, learning_rate=0.15)

epochs = 15
batch_size = 64

print("\nStarting Training Pipeline...")
print("-" * 50)

for epoch in range(epochs):
    # Shuffle dataset at the beginning of each epoch
    permutation = np.random.permutation(num_samples)
    X_shuffled = X_train[permutation]
    Y_shuffled_onehot = Y_train_onehot[permutation]

    # Mini-batch gradient descent loop
    for i in range(0, num_samples, batch_size):
        X_batch = X_shuffled[i : i + batch_size]
        Y_batch_onehot = Y_shuffled_onehot[i : i + batch_size]

        mlp.forward(X_batch)
        mlp.backward(X_batch, Y_batch_onehot)

    # Evaluate performance at the end of the epoch
    train_predictions = mlp.forward(X_train)
    train_loss = compute_loss(train_predictions, Y_train_onehot)
    train_acc = get_accuracy(train_predictions, Y_train_labels)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc*100:.2f}%")

print("-" * 50)
print("Training Complete. Evaluating on Test Set...")

# Final verification on the completely unseen test split
test_predictions = mlp.forward(X_test)
test_loss = compute_loss(test_predictions, Y_test_onehot)
test_acc = get_accuracy(test_predictions, Y_test_labels)

print(f"\n[FINAL RESULTS] Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%")

Q2
CNN takes more time and has more number of parameters it keeps on preserving the spatial characteristc but MLP flatten the pixel into vector in the starting . so CNN has a better accuracy as well.

In [1]:
# ==========================================
# MLP vs CNN on MNIST using PyTorch
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

# ------------------------------------------
# Device Configuration
# ------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

# ------------------------------------------
# Load MNIST Dataset
# ------------------------------------------

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print(f"Training Samples: {len(train_dataset)}")
print(f"Testing Samples: {len(test_dataset)}")

# ------------------------------------------
# MLP Model
# ------------------------------------------

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(28 * 28, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

# ------------------------------------------
# CNN Model
# ------------------------------------------

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ------------------------------------------
# Utility Functions
# ------------------------------------------

def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

def train_model(model, train_loader, epochs=5):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.to(device)

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f}"
        )

    training_time = time.time() - start_time

    return training_time

def evaluate_model(model, test_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy

# ==========================================
# Train MLP
# ==========================================

print("\n" + "="*50)
print("TRAINING MLP")
print("="*50)

mlp = MLP()

mlp_params = count_parameters(mlp)

print(f"MLP Parameters: {mlp_params:,}")

mlp_time = train_model(
    mlp,
    train_loader,
    epochs=5
)

mlp_accuracy = evaluate_model(
    mlp,
    test_loader
)

# ==========================================
# Train CNN
# ==========================================

print("\n" + "="*50)
print("TRAINING CNN")
print("="*50)

cnn = CNN()

cnn_params = count_parameters(cnn)

print(f"CNN Parameters: {cnn_params:,}")

cnn_time = train_model(
    cnn,
    train_loader,
    epochs=5
)

cnn_accuracy = evaluate_model(
    cnn,
    test_loader
)

# ==========================================
# Final Comparison
# ==========================================

print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)

print(f"{'Metric':<20}{'MLP':<20}{'CNN':<20}")
print("-"*60)

print(f"{'Accuracy (%)':<20}{mlp_accuracy:<20.2f}{cnn_accuracy:<20.2f}")

print(f"{'Training Time(s)':<20}{mlp_time:<20.2f}{cnn_time:<20.2f}")

print(f"{'Parameters':<20}{mlp_params:<20,}{cnn_params:<20,}")

print("="*60)

# ------------------------------------------
# Simple Analysis
# ------------------------------------------

print("\nANALYSIS")
print("-"*60)

if cnn_accuracy > mlp_accuracy:
    print("CNN achieved higher accuracy than MLP.")

if cnn_time > mlp_time:
    print("CNN took longer to train due to convolution operations.")
else:
    print("CNN trained faster than MLP.")

print(
    "CNN preserves spatial information through convolutional "
    "layers, making it more effective for image classification."
)

Using Device: cpu


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.11MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 139kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]


Training Samples: 60000
Testing Samples: 10000

TRAINING MLP
MLP Parameters: 235,146
Epoch [1/5] Loss: 0.2783
Epoch [2/5] Loss: 0.1089
Epoch [3/5] Loss: 0.0726
Epoch [4/5] Loss: 0.0516
Epoch [5/5] Loss: 0.0402

TRAINING CNN
CNN Parameters: 421,642
Epoch [1/5] Loss: 0.1856
Epoch [2/5] Loss: 0.0537
Epoch [3/5] Loss: 0.0368
Epoch [4/5] Loss: 0.0271
Epoch [5/5] Loss: 0.0216

FINAL COMPARISON
Metric              MLP                 CNN                 
------------------------------------------------------------
Accuracy (%)        97.77               98.91               
Training Time(s)    73.39               403.92              
Parameters          235,146             421,642             

ANALYSIS
------------------------------------------------------------
CNN achieved higher accuracy than MLP.
CNN took longer to train due to convolution operations.
CNN preserves spatial information through convolutional layers, making it more effective for image classification.


Q3
1) Activation function is used to handle non linear functions. we have a linear function so we dont need to introduce something to cater non linearity.
2) since its a very simple linear function y=x we expect the structure to simple input output layer with weight = 1 and bias =0; y= w*x + b
3) The simple 2 layer input output architecture completely works. Introducing more layers just increases the number of parameters and conditions to be handled for this very simple equations.
4) yes the results are like what we wanted.


In [ ]:
import numpy as np

# 1. Generate synthetic data for y = x
np.random.seed(42)
X_train = np.random.uniform(-10, 10, (1000, 1))
Y_train = X_train  # y = x

# 2. Setup a 1-Layer Network (No hidden layer, no activation)
class IdentityMLP:
    def __init__(self):
        # Initializing with random numbers
        self.w = np.random.randn(1, 1) * 100
        self.b = np.random.randn(1, 1) * 100

    def forward(self, x):
        return np.dot(x, self.w) + self.b

    def train(self, x, y_true, lr=0.001):
        m = x.shape[0]  # Number of samples (1000)
        y_pred = self.forward(x)

        # Derivative of MSE Loss: 2/m * (y_pred - y_true)
        # Dividing by m ensures the gradient represents the AVERAGE error, not the SUMmed error
        dL_dy_pred = 2 * (y_pred - y_true) / m

        dw = np.dot(x.T, dL_dy_pred)
        db = np.sum(dL_dy_pred, axis=0, keepdims=True)

        # Update weights smoothly
        self.w -= lr * dw
        self.b -= lr * db

# 3. Execution Pipeline
model = IdentityMLP()
print(f"Initial Random Weight: {model.w[0][0]:.4f} | Initial Random Bias: {model.b[0][0]:.4f}")
print("Training...")

# Using a safe learning rate with the averaged gradients
for epoch in range(1000):
    model.train(X_train, Y_train, lr=0.01)

print("-" * 50)
print(f"Final Trained Weight:  {model.w[0][0]:.6f} (Expected: 1.000000)")
print(f"Final Trained Bias:    {model.b[0][0]:.6f} (Expected: 0.000000)")

Initial Random Weight: 17.7701 | Initial Random Bias: -133.5344
Training...
--------------------------------------------------
Final Trained Weight:  1.000000 (Expected: 1.000000)
Final Trained Bias:    -0.000000 (Expected: 0.000000)


Q4
Even with a combination of many neurons layers with linear nodes th eoutput is still a linear function . So we need activation function.
Error using activation Relu function is less.

In [ ]:
import numpy as np

# 1. Generate synthetic data for a parabola: y = x^2
np.random.seed(42)
X_train = np.random.uniform(-2, 2, (2000, 1))  # Inputs between -2 and 2
Y_train = X_train ** 2                         # Target is x squared

# ===================================================
# EXPERIMENT A: DEEP NETWORK WITH NO ACTIVATION (LINEAR)
# ===================================================
class LinearNetwork:
    def __init__(self):
        # Architecture: 1 -> 10 -> 1
        self.W1 = np.random.randn(1, 10) * 0.1
        self.b1 = np.zeros((1, 10))
        self.W2 = np.random.randn(10, 1) * 0.1
        self.b2 = np.zeros((1, 1))

    def forward(self, x):
        self.Z1 = np.dot(x, self.W1) + self.b1
        self.A1 = self.Z1  # NO ACTIVATION FUNCTION (Purely Linear)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        return self.Z2

    def train(self, x, y_true, lr=0.01):
        m = x.shape[0]
        y_pred = self.forward(x)

        # Backpropagation
        dZ2 = 2 * (y_pred - y_true) / m
        dW2 = np.dot(self.A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * 1  # Derivative of linear activation is 1
        dW1 = np.dot(x.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Updates
        self.W2 -= lr * dW2; self.b2 -= lr * db2
        self.W1 -= lr * dW1; self.b1 -= lr * db1

# ===================================================
# EXPERIMENT B: DEEP NETWORK WITH RELU ACTIVATION
# ===================================================
class ReLUActivationNetwork:
    def __init__(self):
        # Architecture: 1 -> 10 -> 1
        self.W1 = np.random.randn(1, 10) * 0.5
        self.b1 = np.zeros((1, 10))
        self.W2 = np.random.randn(10, 1) * 0.5
        self.b2 = np.zeros((1, 1))

    def forward(self, x):
        self.Z1 = np.dot(x, self.W1) + self.b1
        self.A1 = np.maximum(0, self.Z1)  # ACTIVATION FUNCTION ENABLED!
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        return self.Z2

    def train(self, x, y_true, lr=0.01):
        m = x.shape[0]
        y_pred = self.forward(x)

        # Backpropagation
        dZ2 = 2 * (y_pred - y_true) / m
        dW2 = np.dot(self.A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (self.Z1 > 0)  # Derivative of ReLU
        dW1 = np.dot(x.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Updates
        self.W2 -= lr * dW2; self.b2 -= lr * db2
        self.W1 -= lr * dW1; self.b1 -= lr * db1

# ===================================================
# RUNNING THE EXPERIMENTS
# ===================================================

print("Running Experiment A (No Activation Function)...")
linear_model = LinearNetwork()
for epoch in range(2000):
    linear_model.train(X_train, Y_train, lr=0.05)
linear_pred = linear_model.forward(X_train)
linear_loss = np.mean((linear_pred - Y_train) ** 2)

print("\nRunning Experiment B (With ReLU Activation Function)...")
relu_model = ReLUActivationNetwork()
for epoch in range(5000):
    relu_model.train(X_train, Y_train, lr=0.05)
relu_pred = relu_model.forward(X_train)
relu_loss = np.mean((relu_pred - Y_train) ** 2)

print("-" * 60)
print(f"[RESULTS] Final Loss WITHOUT Activation (Linear): {linear_loss:.4f}")
print(f"[RESULTS] Final Loss WITH ReLU Activation:        {relu_loss:.4f}")
print("-" * 60)

Running Experiment A (No Activation Function)...

Running Experiment B (With ReLU Activation Function)...
------------------------------------------------------------
[RESULTS] Final Loss WITHOUT Activation (Linear): 1.4415
[RESULTS] Final Loss WITH ReLU Activation:        0.0016
------------------------------------------------------------


Q5
Cross entropy with softmax works better because it work by penalizes if the probability predicted for the correct classification is low.All the training is done based on probability.
MSE with tanh does not calculate probabilities . it works on how much the output is exactly like what it should be.